First pass at making a baseline generator (spend, channel, impressions)

Based on the marketing funnel, we can create the following casual diagram:

Spend -> Impressions -> (Conversions) -> Revenue 

* Conversions represents those who see the ad and proceed to purchase the product. 

Since this is relatively basic we will assume that: 

- Spend is randomized since we do not have a specific business we are modeling
- Impressions are a transformation of the spend with some noise
- Revenue consists of baseline demand + incremental media channel effect + noise

Assumptions: 
- Values are always positive
- Week-to-week values can jump a lot (we can add smoothness later)
- For now this means weeks are independent of each other 

Let’s say we want to model an ROI of 2. Based off of the monthly_mocha sample dataset  I created some range estimates keeping the ROI in mind:

Spend (10,000 - 100,000) 
Impressions (100,000 - 1,000,000)
Revenue (20,000 - 200,000)

Noise terms are iid normal with zero mean and standard deviation equal to 5% the expected value of the corresponding range. (*Note: the standard deviation was an arbitrary choice)

Calculations for each week: 
* weekly_spend : random value from the given range
* weekly_impressions : weekly_spend * channel_efficiency + noise 
* weekly_revenue:  baseline_revenue +  (weekly_impressions * channel_conversions) *  revenue_per_conversion + noise

- Channel_efficiency refers to the impressions per dollar spent
- Baseline_revenue refers to the demand without any marketing intervention (arbitrary choice) 
- Channel_conversions refers to how many people make it down the marketing funnel and purchase the - product (arbitrary choice)
- Revenue_per_conversion refers to the price of the product 


In [8]:
# Spend (10,000 - 100,000)
# Impressions (100,000 - 1,000,000)
# Revenue (20,000 - 200,000)

import numpy as np
import random
import pandas as pd
random.seed(99)

spend = []
impressions = []
revenue = []

#impressions per dollar spend
channel_efficieny = 10

# 10% of users see the ad -> and buy
channel_conversions = 0.1
revenue_per_conversion = 2
baseline_revenue = 20000

noise = 0

weeks = 52
for i in range (1,weeks+1):
    weekly_spend = random.randrange(10000, 100000)
    spend.append(weekly_spend)
    weekly_impresisons = weekly_spend * channel_efficieny + np.random.normal(loc=0, scale=25000)
    impressions.append(weekly_impresisons)
    weekly_revenue = baseline_revenue + (weekly_impresisons * channel_conversions) * revenue_per_conversion + np.random.normal(loc=0, scale=5500)
    revenue.append(weekly_revenue)

df = pd.DataFrame(data = {'Spend':spend, 'Impressions':impressions, 'Revenue':revenue})
df

,Spend,Impressions,Revenue
0,62950,593739.976745,134402.905108
1,59906,596098.566036,140704.655657
2,36224,353573.765665,95346.534363
3,88569,915868.802997,201163.928353
4,33435,283977.817874,80717.767994
5,40180,358606.948074,84153.262636
6,42562,405485.589810,91659.962867
7,27464,224146.810506,63859.545112
8,21348,213230.514893,59707.830064
9,42918,443115.724763,97003.311999
